# Filter vs Vector: EMA, DEMA, TEMA

Compares `cdef class` filter approach (one loop, no intermediate allocations)
against the current `calc_*` vector functions (multiple passes, intermediate arrays).

Three filter variants:
- **base class** — `process()` on base `Filter`, calls `self.update()` via virtual dispatch
- **no base class** — each class has its own `process()`, `cdef _update()` is a direct C call
- **inline** — same as no base class but `cdef inline _update()`, body inlined into the loop

In [ ]:
import timeit
import numpy as np

%load_ext cython

from mintalib import core
from mintalib.samples import sample_prices

prices = sample_prices()
close = prices.close.values.astype(float)
print(f"Rows: {len(close):,}")

In [ ]:
%%cython -c=-Wno-unreachable-code
# cython: boundscheck=False, wraparound=False, cdivision=True, nonecheck=False
# Variant 1: with base class — virtual dispatch in process()

import numpy as np
from libc.math cimport isnan

cdef double NAN = float('nan')


cdef class Filter:
    cpdef double update(self, double x) except *:
        return NAN

    cpdef process(self, double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.full(size, NAN)
        cdef double[:] output = result
        cdef long i = 0
        for i in range(size):
            output[i] = self.update(xs[i])
        return result


cdef class EmaFilter(Filter):
    cdef readonly int period
    cdef double alpha, value
    cdef long count

    def __init__(self, int period):
        self.period = period
        self.alpha = 2.0 / (period + 1.0)
        self.value = NAN
        self.count = 0

    cpdef void reset(self):
        self.value = NAN
        self.count = 0

    cpdef double update(self, double x) except *:
        if isnan(x):
            self.reset()
            return NAN
        self.count += 1
        if isnan(self.value):
            self.value = x
        else:
            self.value += self.alpha * (x - self.value)
        if self.count >= self.period:
            return self.value
        return NAN


cdef class DemaFilter(Filter):
    cdef readonly int period
    cdef EmaFilter ema1, ema2

    def __init__(self, int period):
        self.period = period
        self.ema1 = EmaFilter(period)
        self.ema2 = EmaFilter(period)

    cpdef double update(self, double x) except *:
        cdef double e1 = self.ema1.update(x)
        cdef double e2 = self.ema2.update(e1)
        if isnan(e1) or isnan(e2):
            return NAN
        return 2.0 * e1 - e2


cdef class TemaFilter(Filter):
    cdef readonly int period
    cdef EmaFilter ema1, ema2, ema3

    def __init__(self, int period):
        self.period = period
        self.ema1 = EmaFilter(period)
        self.ema2 = EmaFilter(period)
        self.ema3 = EmaFilter(period)

    cpdef double update(self, double x) except *:
        cdef double e1 = self.ema1.update(x)
        cdef double e2 = self.ema2.update(e1)
        cdef double e3 = self.ema3.update(e2)
        if isnan(e1) or isnan(e2) or isnan(e3):
            return NAN
        return 3.0 * e1 - 3.0 * e2 + e3


print("Done (base class)!")

In [ ]:
%%cython -c=-Wno-unreachable-code
# cython: boundscheck=False, wraparound=False, cdivision=True, nonecheck=False
# Variant 2: no base class — cdef _update(), direct C call

import numpy as np
from libc.math cimport isnan

cdef double NAN2 = float('nan')


cdef class EmaFilter2:
    cdef readonly int period
    cdef double alpha, value
    cdef long count

    def __init__(self, int period):
        self.period = period
        self.alpha = 2.0 / (period + 1.0)
        self.value = NAN2
        self.count = 0

    cdef void reset(self):
        self.value = NAN2
        self.count = 0

    cdef double _update(self, double x) noexcept:
        if isnan(x):
            self.reset()
            return NAN2
        self.count += 1
        if isnan(self.value):
            self.value = x
        else:
            self.value += self.alpha * (x - self.value)
        if self.count >= self.period:
            return self.value
        return NAN2

    def process(self, double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.full(size, NAN2)
        cdef double[:] output = result
        cdef long i = 0
        for i in range(size):
            output[i] = self._update(xs[i])
        return result


cdef class DemaFilter2:
    cdef readonly int period
    cdef EmaFilter2 ema1, ema2

    def __init__(self, int period):
        self.period = period
        self.ema1 = EmaFilter2(period)
        self.ema2 = EmaFilter2(period)

    def process(self, double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.full(size, NAN2)
        cdef double[:] output = result
        cdef double e1, e2
        cdef long i = 0
        for i in range(size):
            e1 = self.ema1._update(xs[i])
            e2 = self.ema2._update(e1)
            if not isnan(e1) and not isnan(e2):
                output[i] = 2.0 * e1 - e2
        return result


cdef class TemaFilter2:
    cdef readonly int period
    cdef EmaFilter2 ema1, ema2, ema3

    def __init__(self, int period):
        self.period = period
        self.ema1 = EmaFilter2(period)
        self.ema2 = EmaFilter2(period)
        self.ema3 = EmaFilter2(period)

    def process(self, double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.full(size, NAN2)
        cdef double[:] output = result
        cdef double e1, e2, e3
        cdef long i = 0
        for i in range(size):
            e1 = self.ema1._update(xs[i])
            e2 = self.ema2._update(e1)
            e3 = self.ema3._update(e2)
            if not isnan(e1) and not isnan(e2) and not isnan(e3):
                output[i] = 3.0 * e1 - 3.0 * e2 + e3
        return result


print("Done (no base class)!")

In [ ]:
%%cython -c=-Wno-unreachable-code
# cython: boundscheck=False, wraparound=False, cdivision=True, nonecheck=False
# Variant 3: no base class — cdef inline _update(), inlined into the loop

import numpy as np
from libc.math cimport isnan

cdef double NAN3 = float('nan')


cdef class EmaFilter3:
    cdef readonly int period
    cdef double alpha, value
    cdef long count

    def __init__(self, int period):
        self.period = period
        self.alpha = 2.0 / (period + 1.0)
        self.value = NAN3
        self.count = 0

    cdef inline void reset(self):
        self.value = NAN3
        self.count = 0

    cdef inline double _update(self, double x) noexcept:
        if isnan(x):
            self.reset()
            return NAN3
        self.count += 1
        if isnan(self.value):
            self.value = x
        else:
            self.value += self.alpha * (x - self.value)
        if self.count >= self.period:
            return self.value
        return NAN3

    def process(self, double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.full(size, NAN3)
        cdef double[:] output = result
        cdef long i = 0
        for i in range(size):
            output[i] = self._update(xs[i])
        return result


cdef class DemaFilter3:
    cdef readonly int period
    cdef EmaFilter3 ema1, ema2

    def __init__(self, int period):
        self.period = period
        self.ema1 = EmaFilter3(period)
        self.ema2 = EmaFilter3(period)

    def process(self, double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.full(size, NAN3)
        cdef double[:] output = result
        cdef double e1, e2
        cdef long i = 0
        for i in range(size):
            e1 = self.ema1._update(xs[i])
            e2 = self.ema2._update(e1)
            if not isnan(e1) and not isnan(e2):
                output[i] = 2.0 * e1 - e2
        return result


cdef class TemaFilter3:
    cdef readonly int period
    cdef EmaFilter3 ema1, ema2, ema3

    def __init__(self, int period):
        self.period = period
        self.ema1 = EmaFilter3(period)
        self.ema2 = EmaFilter3(period)
        self.ema3 = EmaFilter3(period)

    def process(self, double[:] xs):
        cdef long size = xs.shape[0]
        cdef object result = np.full(size, NAN3)
        cdef double[:] output = result
        cdef double e1, e2, e3
        cdef long i = 0
        for i in range(size):
            e1 = self.ema1._update(xs[i])
            e2 = self.ema2._update(e1)
            e3 = self.ema3._update(e2)
            if not isnan(e1) and not isnan(e2) and not isnan(e3):
                output[i] = 3.0 * e1 - 3.0 * e2 + e3
        return result


print("Done (inline)!")

In [ ]:
# Sanity check
period = 20

def check(a, b, label):
    a, b = np.asarray(a)[200:], np.asarray(b)[200:]
    mask = ~(np.isnan(a) | np.isnan(b))
    ok = np.allclose(a[mask], b[mask], rtol=1e-10)
    print(f"{label}: {'OK' if ok else 'MISMATCH'}")

check(EmaFilter(period).process(close),   core.calc_ema(close, period),  "EMA  (base)")
check(EmaFilter2(period).process(close),  core.calc_ema(close, period),  "EMA  (no base)")
check(EmaFilter3(period).process(close),  core.calc_ema(close, period),  "EMA  (inline)")
check(DemaFilter(period).process(close),  core.calc_dema(close, period), "DEMA (base)")
check(DemaFilter2(period).process(close), core.calc_dema(close, period), "DEMA (no base)")
check(DemaFilter3(period).process(close), core.calc_dema(close, period), "DEMA (inline)")
check(TemaFilter(period).process(close),  core.calc_tema(close, period), "TEMA (base)")
check(TemaFilter2(period).process(close), core.calc_tema(close, period), "TEMA (no base)")
check(TemaFilter3(period).process(close), core.calc_tema(close, period), "TEMA (inline)")

In [ ]:
# Benchmark
def bench(fn, repeat=5, number=10):
    times = timeit.repeat(fn, repeat=repeat, number=number)
    return min(times) / number

def fmt(t):
    return f"{t * 1000:.3f} ms"

period = 20
cases = [
    ("EMA",  EmaFilter(period).process,   EmaFilter2(period).process,   EmaFilter3(period).process,   lambda: core.calc_ema(close, period)),
    ("DEMA", DemaFilter(period).process,  DemaFilter2(period).process,  DemaFilter3(period).process,  lambda: core.calc_dema(close, period)),
    ("TEMA", TemaFilter(period).process,  TemaFilter2(period).process,  TemaFilter3(period).process,  lambda: core.calc_tema(close, period)),
]

print(f"Rows: {len(close):,}  period={period}\n")
print(f"{'':6}  {'base':>10}  {'no base':>10}  {'inline':>10}  {'vector':>10}")
print("-" * 58)

for name, f_base, f_nobase, f_inline, f_vector in cases:
    t_base   = bench(lambda f=f_base:   f(close))
    t_nobase = bench(lambda f=f_nobase: f(close))
    t_inline = bench(lambda f=f_inline: f(close))
    t_vector = bench(f_vector)
    print(f"{name:<6}  {fmt(t_base):>10}  {fmt(t_nobase):>10}  {fmt(t_inline):>10}  {fmt(t_vector):>10}")